# Conformal prediction intervals

Conformal prediction uses calibration residuals to turn a fitted prediction
$\hat\mu(x)$ into an interval for the next observation. Conformal prediction constructs such an interval with a
finite-sample marginal coverage guarantee for split conformal and CQR when
the rows are exchangeable and the fitting rule is symmetric
{cite}`vovk2005algorithmic`. The specification, transformations,
hyperparameters, and stopping rule must be fixed without using calibration
responses. CV+ has a different lower bound, described below.


## The marginal guarantee

For a target level $1-\alpha$, a conformal procedure returns a set
$C(x)$ with

$$\mathbb{P}\big(Y_{n+1} \in C(X_{n+1})\big) \ge 1-\alpha,$$

where the probability is over the joint draw of the training data and the
test point. The coverage is *marginal*: averaged over $X$, not guaranteed
at a fixed $x$. A constant-width interval attains the level overall while
over-covering where the noise is small and under-covering where it is
large.


## Split conformal

Split the data into a training half and a calibration half
{cite}`lei2018distribution`. Fit $\hat\mu$ on the training half. On the
calibration set of size $n_c$ compute the absolute residuals
$R_i = |Y_i - \hat\mu(X_i)|$. The interval at a new point is

$$C(x) = \hat\mu(x) \pm Q,\qquad Q = R_{(k)},\quad
k = \lceil (n_c+1)(1-\alpha)\rceil,$$

the $k$-th smallest calibration residual. The $+1$ in $k$ gives finite-sample coverage under exchangeability. If
$k > n_c$, the requested level is unattainable and the interval is infinite.
The procedure requires one extra fit and gives an interval of constant width
in $x$.


## CV+

Split conformal spends half the data on calibration. CV+ uses $K$-fold
cross-validation instead {cite}`barber2021predictive`: every observation is
held out once, giving an out-of-fold residual $R_i$ and an out-of-fold
prediction $\hat\mu_{-k(i)}(x)$. The interval endpoints are order
statistics of the leave-fold-out predictions shifted by those residuals,
so all $n$ observations inform both the fit and the calibration. The cost
is $K$ fits. CV+ does not inherit the split-conformal $1-\alpha$ bound.
For balanced folds, `ConformalInterval.guaranteed_coverage` reports the
finite-sample lower bound from Barber et al.; it is generally below the
nominal `level`. Fold-specific predictions vary with $x$, so CV+ width
need not be constant.


## Conformalized quantile regression

Constant-width intervals are inefficient under heteroscedasticity.
Conformalized quantile regression (CQR) fits two quantile models at levels
$\alpha/2$ and $1-\alpha/2$, then conformalizes the gap
{cite}`romano2019conformalized`. The calibration score is how far outside
the predicted band an observation fell,

$$E_i = \max\{\hat q_{\alpha/2}(X_i) - Y_i,\; Y_i - \hat q_{1-\alpha/2}(X_i)\},$$

and the band is widened (or narrowed) by the conformal quantile of the
$E_i$. The result keeps the marginal guarantee and inherits the varying
width of the underlying quantile fits. mboostrs fits the two quantile
models with the `Quantile` family, so CQR reuses the same engine.


## Coverage on heteroscedastic data

The noise scale below grows with $x$. The simulation measures empirical
fresh-sample coverage; it does not replace a theorem. Split conformal has
constant width, while CV+ and CQR can vary with $x$.


In [1]:
import numpy as np
import mboostrs as mb

rng = np.random.default_rng(0)
def heteroscedastic(n):
    x = rng.uniform(-2, 2, n)
    scale = 0.1 + 0.5 * (x + 2.0)
    return {"x": x, "y": np.sin(x) + rng.normal(scale=scale, size=n)}

train, test = heteroscedastic(600), heteroscedastic(600)
fit = mb.gamboost("y ~ bbs(x, df=4)", data=train, family=mb.Gaussian(),
                  control=mb.boost_control(200, 0.1))

rows = []
for method in ("split", "cv+", "cqr"):
    ci = fit.conformal_interval(newdata=test, level=0.9, method=method,
                                seed=1, n_folds=5)
    width = ci.upper - ci.lower
    rows.append({
        "method": method,
        "coverage": round(ci.empirical_coverage(test["y"]), 3),
        "mean_width": round(float(np.mean(width)), 3),
        "width_varies_with_x": round(float(np.corrcoef(test["x"], width)[0, 1]), 2),
    })
rows


/Users/fcela/src/public_github/mboostrs/python/mboostrs/gamboost.py:3082: UserWarning: optimal mstop 200 is within 5% of the search cap 200; the search range was exhausted — consider increasing mstop and re-running cvrisk.
  result = CvRiskResult(
/Users/fcela/src/public_github/mboostrs/python/mboostrs/gamboost.py:3082: UserWarning: optimal mstop 200 is within 5% of the search cap 200; the search range was exhausted — consider increasing mstop and re-running cvrisk.
  result = CvRiskResult(


[{'method': 'split',
  'coverage': 0.888,
  'mean_width': 4.048,
  'width_varies_with_x': -0.0},
 {'method': 'cv+',
  'coverage': 0.9,
  'mean_width': 4.165,
  'width_varies_with_x': 0.34},
 {'method': 'cqr',
  'coverage': 0.853,
  'mean_width': 3.375,
  'width_varies_with_x': 1.0}]

`split` is constant-width. CV+ varies through its fold-specific predictions,
and `cqr` widens as the fitted conditional quantiles separate. All three have
empirical coverage near the 0.9 target on this fresh sample; the result
objects report their different theoretical bounds separately.


## Response transformations

A response written as `center(y)` or `scale(y)` is modelled on a transformed scale. The fit learns a centre or a scale from the rows it uses, and its predictions come back to the original units of `y` with those same constants. The conformal methods use this rule for every calibration score: the observed responses and the predictions are both in the original units of `y`.

Calibration scores must compare responses and predictions in the same units. If the calibration responses were centred with one set of constants and the predictions with another, the band would miss by the difference between them. The calibration responses are therefore never transformed with the full-data constants when the refit used different ones.

For split conformal and CV+, the result does not depend on the transform. The predictions are mapped back to the original units with the refit's own constants, so a centred or scaled model gives the same interval as the untransformed model, up to the randomness of the fold split. The measured values are in the last section of this page.

## Step count for the quantile models

CQR fits two quantile models, and each needs a number of boosting steps. With `mstop="tune"` the step count is chosen on the proper-training rows only, never on the calibration rows, so the stopping decision does not use calibration responses.

The search runs as follows. It starts at the larger of the model's own step count and 1600. Five-fold cross-validation on the proper-training rows gives a risk for every step count on the grid. If the lowest risk lies in the last 5% of the grid, the risk has not turned, so the grid is doubled and the search repeats. The search stops when the lowest risk is more than 5% short of the grid end, or when the grid reaches the cap of 10,000 steps. At the cap, a warning reports that the risk was still falling there. The reported step count is the step with the lowest risk on the final grid, and it is never zero.

A centred or scaled response adds one condition. The constants are learned on the proper-training rows, and the same constants must apply to all five folds. If each fold relearned them from its own rows, the fold risks would be measured on different scales and could not be compared, so their average would combine different loss scales. The fold-fixed rule uses one set of constants for every fold. This was previously refused for CQR, and it is now allowed for every tuned method (#663).

The search has a cost. Each quantile model is cross-validated at every grid end, so a tuned CQR fit at 10,000 observations takes about 2 seconds, against about 0.1 to 0.2 seconds for the same fit with a fixed step count. Most of the time is the held-out prediction along each fold's path. The search is kept because it sets the step count from the training rows, which the coverage construction relies on. To skip it, pass an integer `mstop` to `conformal_interval`: each quantile model is then fitted at that step count, and the search does not run. The choice of step count is then the caller's, and the interval's guarantee rests on that choice being fixed before the calibration rows are seen.

## Measured coverage and width with transformed responses

The interval study in `docs/book/benchmarks/interval_calibration` was rerun with the response unchanged, centred and scaled, and with the step count tuned by cross-validation. Each cell is the mean over 20 seeds for 1000 training rows and 2000 test rows, at a 90% target. Coverage is the share of test responses inside the interval, and width is the mean interval width. Results are in `transform_tuning.json` in the same folder.

| Process | Response | Split: coverage / width | CV+: coverage / width | CQR: coverage / width |
|---|---|---|---|---|
| Homoscedastic | `y` | 0.901 / 3.34 | 0.904 / 3.35 | 0.901 / 3.47 |
| Homoscedastic | `center(y)` | 0.901 / 3.34 | 0.904 / 3.35 | 0.901 / 3.48 |
| Homoscedastic | `scale(y)` | 0.901 / 3.34 | 0.904 / 3.35 | 0.902 / 3.48 |
| Heteroscedastic | `y` | 0.898 / 3.82 | 0.899 / 3.82 | 0.900 / 3.63 |
| Heteroscedastic | `center(y)` | 0.898 / 3.82 | 0.899 / 3.82 | 0.901 / 3.63 |
| Heteroscedastic | `scale(y)` | 0.898 / 3.82 | 0.899 / 3.82 | 0.898 / 3.59 |
| Heavy-tailed | `y` | 0.900 / 2.76 | 0.901 / 2.74 | 0.904 / 3.05 |
| Heavy-tailed | `center(y)` | 0.901 / 2.76 | 0.901 / 2.74 | 0.902 / 3.04 |
| Heavy-tailed | `scale(y)` | 0.901 / 2.76 | 0.901 / 2.74 | 0.902 / 3.01 |

Competitor widths at nominal coverage, from the same study with untransformed responses and fixed settings. The competitors are not rerun here.

| Process | CatBoost CQR | LightGBM CQR | sklearn gradient boosting CQR |
|---|---|---|---|
| Homoscedastic | 3.84 | 3.66 | 3.69 |
| Heteroscedastic | 3.93 | 3.75 | 3.84 |
| Heavy-tailed | 3.43 | 3.22 | 3.29 |

Every refused case in the earlier run is now computed, and none of the 540 runs was refused. With a transformed response, CQR's width is within about 1% of its untransformed width on each process, and its coverage stays at the target. At this nominal level, mboostrs CQR is narrower than all three competitors on every process.

The comparison has limits. The mboostrs models are given the true specification, `bbs(x1, df=4) + bols(x2)`, while the competitors receive the raw features with fixed settings. The study does not measure run time, so the gap in CQR run time reported in #578 is not addressed here.